# Challenge 1 — GHZ optimisé pour une vraie puce IBM

**Idée :** les deux algorithmes donnés (cascade linéaire et arbre binaire) ignorent la puce.
La cascade est trop profonde, et l'arbre relie des qubits éloignés, ce qui oblige à ajouter des SWAP.
Ici, on construit un **arbre parallèle qui ne suit que les vrais câbles de la puce** :

| | Cascade | Arbre binaire | **Notre version** |
|---|---|---|---|
| Profondeur | ≈ N | log₂N (idéal) | **≈ log₂N + 1 ou 2** |
| SWAP sur heavy-hex | 0 | beaucoup | **0** |
| Choix des qubits | au hasard | au hasard | **les moins bruités du jour** |
| Choix de N | fixé | fixé | **N qui maximise le score** |

Étapes :
1. lire la calibration du jour de la puce IBM (erreur de chaque lien et de chaque lecture) ;
2. faire grandir le GHZ couche par couche, depuis chaque qubit de départ possible, en gardant le meilleur ;
3. prédire le score pour chaque N, puis lancer les 3 meilleurs N sur la vraie machine ;
4. corriger les erreurs de lecture, calculer le score et soumettre le meilleur.

In [ ]:
# %pip install -q qiskit qiskit-ibm-runtime matplotlib requests
import math
import requests
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile, qasm2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

## 1. Connexion à IBM Quantum
Mets ta clé API IBM Quantum (page d'accueil de quantum.cloud.ibm.com). Si tu l'as déjà enregistrée avec `save_account`, laisse `TOKEN = None`.

In [ ]:
TOKEN = "AEcaBNxDgItWaGdq3x1b7uzglqmOJWNFRUlbaCjxCzy9"   # cle API IBM Quantum (ne partage pas ce fichier)
INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/5ed2b12c458c4af986180f529802c147:ba1c8880-000c-4af7-8aa6-628b7cee2734::"
BACKEND_NAME = None   # ex: "ibm_fez" ; None = la machine la moins chargee

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN, instance=INSTANCE) if TOKEN \
    else QiskitRuntimeService()
backend = service.backend(BACKEND_NAME) if BACKEND_NAME else \
    service.least_busy(operational=True, simulator=False, min_num_qubits=100)
print("Machine :", backend.name, "-", backend.num_qubits, "qubits")

## 2. L'algorithme
- **`hardware_info`** lit la calibration : l'erreur de chaque lien CNOT, l'erreur de lecture et le T1 de chaque qubit. Les liens et qubits cassés sont exclus.
- **`grow_tree`** : à chaque couche, *tous* les qubits déjà intriqués font en parallèle une CNOT vers un voisin libre. C'est l'arbre binaire, mais seulement sur les câbles de la puce, donc **0 SWAP**. Les liens les moins bruités passent en premier.
- **`estimate_correlation`** prédit C. Astuce : le score ne regarde que les comptages `00…0`/`11…1`, donc **seules les erreurs qui retournent un bit (X/Y) comptent**. Les erreurs de phase (Z) sont invisibles.
- **`search`** essaie chaque qubit comme point de départ, avec plusieurs réglages, et garde le meilleur plan pour chaque N.

In [ ]:
import math
import random

from qiskit import QuantumCircuit


def hardware_info(backend):
    """Lit la calibration du jour : erreur de chaque lien, erreur de lecture, T1 de chaque qubit."""
    t = backend.target
    n = t.num_qubits
    two_q = next(g for g in ("cz", "ecr", "cx") if g in t.operation_names)

    def props(op, qargs):
        return t[op].get(qargs) if op in t.operation_names else None

    sx, ro, t1 = {}, {}, {}
    for q in range(n):
        p = props("sx", (q,))
        sx[q] = p.error if (p and p.error is not None) else 3e-4
        p = props("measure", (q,))
        ro[q] = p.error if (p and p.error is not None) else 2e-2
        qp = t.qubit_properties[q] if t.qubit_properties else None
        t1[q] = qp.t1 if (qp is not None and qp.t1) else 150e-6

    edges, layer_time = {}, None
    for (a, b), p in t[two_q].items():
        if p is None or p.error is None:
            continue
        e = p.error + 2 * sx[b]          # CX = H . CZ . H  -> erreur 2 qubits + 2 portes sx
        if e >= 0.1:                      # lien casse ou mal calibre : on ne l'utilise jamais
            continue
        key = (min(a, b), max(a, b))
        edges[key] = min(edges.get(key, 1.0), e)
        if layer_time is None and p.duration:
            layer_time = p.duration + 2 * 40e-9
    nbrs = {q: [] for q in range(n)}
    for a, b in edges:
        nbrs[a].append(b)
        nbrs[b].append(a)
    bad = {q for q in range(n) if ro[q] > 0.1}  # qubits a la lecture trop mauvaise : exclus
    return dict(n=n, edges=edges, nbrs=nbrs, ro=ro, t1=t1, bad=bad, layer_time=layer_time or 150e-9)


def edge_err(info, a, b):
    return info["edges"][(min(a, b), max(a, b))]


def grow_tree(info, root, w_future=0.003, rng=None):
    """Fait grandir le GHZ depuis `root`, couche par couche, jusqu'a ne plus pouvoir.

    A chaque couche, TOUS les qubits deja intriques font en parallele une CNOT vers un
    voisin libre (comme l'arbre binaire), mais seulement vers un voisin physique (0 SWAP).
    Les liens les moins bruites passent en premier ; `w_future` favorise les voisins qui
    ont encore des voisins libres, pour garder un front large et donc une faible profondeur.
    """
    nbrs, ro, bad = info["nbrs"], info["ro"], info["bad"]
    ent_set = {root}
    layers = []
    while True:
        cands = []
        for u in ent_set:
            for v in nbrs[u]:
                if v in ent_set or v in bad:
                    continue
                free = sum(1 for w in nbrs[v] if w not in ent_set and w != u and w not in bad)
                cost = -math.log(1 - edge_err(info, u, v)) - math.log(1 - ro[v]) - w_future * free
                if rng is not None:
                    cost += rng.uniform(0, 0.003)
                cands.append((cost, u, v))
        if not cands:
            return layers
        cands.sort()
        used, layer = set(), []
        for cost, u, v in cands:
            if u not in used and v not in used:
                layer.append((u, v))  # deja trie du moins cher au plus cher
                used.update((u, v))
        ent_set.update(v for _, v in layer)
        layers.append(layer)


def truncate(root, layers, n):
    """Garde les n premiers qubits : couches completes puis les liens les moins chers de la derniere."""
    ent, out = [root], []
    for layer in layers:
        room = n - len(ent)
        if room <= 0:
            break
        part = layer[:room]
        out.append(part)
        ent.extend(v for _, v in part)
    return (ent, out) if len(ent) == n else None


def estimate_correlation(info, ent, layers):
    """Estimation de C = P(00..0) + P(11..1).

    Seules les erreurs qui retournent un bit (X ou Y) comptent : C ne regarde que les
    comptages, donc une erreur de phase (Z) est invisible pour le score.
    """
    logc = 0.0
    seen = {ent[0]}
    for layer in layers:
        busy = set()
        for u, v in layer:
            logc += math.log(1 - edge_err(info, u, v) * 12 / 15)  # 12 des 15 Paulis a 2 qubits contiennent X/Y
            busy.update((u, v))
        for q in seen - busy:  # qubit deja intrique qui attend : il peut se relaxer 1 -> 0 (T1)
            logc += math.log(1 - 0.5 * (1 - math.exp(-info["layer_time"] / info["t1"][q])))
        seen.update(v for _, v in layer)
    for q in ent:
        logc += math.log(1 - info["ro"][q])
    return math.exp(logc)


def challenge_score(n, corr, depth, swaps=0, exponent=2.0):
    """Formule du challenge (exposant 2 dans preview_challenge_score, 1.5 dans l'enonce)."""
    rand = 1.0 / 2 ** (n - 1)
    if corr <= rand:
        return 0
    f = ((corr - rand) / (1 - rand)) ** exponent
    dmin = 1 + math.ceil(math.log2(n))
    return round((1000 + 50 * (n * n - 4)) * f / (1 + 0.25 * swaps) * dmin / max(depth, dmin))


def search(info, n_min=8, n_max=None, seed=0):
    """Essaie chaque qubit comme racine et plusieurs reglages ; garde le meilleur plan pour chaque N."""
    rng = random.Random(seed)
    n_max = n_max or info["n"]
    roots = [q for q in range(info["n"]) if q not in info["bad"] and len(info["nbrs"][q]) >= 2]
    best = {}
    for root in roots:
        for w, r in ((0.0, None), (0.003, None), (0.01, None), (0.003, rng)):
            layers = grow_tree(info, root, w, r)
            total = 1 + sum(len(l) for l in layers)
            for n in range(n_min, min(n_max, total) + 1):
                ent, lay = truncate(root, layers, n)
                c = estimate_correlation(info, ent, lay)
                s = challenge_score(n, c, 1 + len(lay))
                if n not in best or s > best[n]["score"]:
                    best[n] = dict(score=s, corr=c, depth=1 + len(lay), ent=ent, layers=lay)
    return best


def build_ghz(ent, layers, measure=True):
    """Circuit a N qubits : le qubit i du circuit est pose sur le qubit physique ent[i]."""
    idx = {p: i for i, p in enumerate(ent)}
    qc = QuantumCircuit(len(ent), len(ent) if measure else 0, name=f"GHZ{len(ent)}")
    qc.h(0)
    for layer in layers:
        for u, v in layer:
            qc.cx(idx[u], idx[v])
    if measure:
        qc.measure(range(len(ent)), range(len(ent)))
    return qc

## 3. Recherche du meilleur N sur la calibration du jour
Le score de base monte en N², mais la corrélation C baisse à chaque qubit ajouté : il existe un **N optimal**. La recherche prend environ 1 minute.

In [ ]:
info = hardware_info(backend)
print(f"{len(info['edges'])} liens utilisables, {len(info['bad'])} qubits exclus")
best = search(info)

ns = sorted(best)
plt.figure(figsize=(8, 3.5))
plt.plot(ns, [best[n]["score"] for n in ns])
plt.xlabel("N (qubits)"); plt.ylabel("score prédit"); plt.title(f"Score prédit sur {backend.name}")
plt.grid(alpha=.3); plt.show()

candidates = sorted(ns, key=lambda n: -best[n]["score"])[:1]
# on teste aussi autour de l'optimum : la prediction n'est qu'une estimation
for n in (candidates[0] - 6, candidates[0] + 8):
    if n in best:
        candidates.append(n)
for n in candidates:
    b = best[n]
    print(f"N={n:3d}  profondeur={b['depth']} (min {1 + math.ceil(math.log2(n))})  C prédit={b['corr']:.3f}  score prédit={b['score']:,}")

## 4. Comparaison avec les algorithmes donnés
Pour le même N, on transpile la cascade et l'arbre du notebook sur la vraie carte de la puce.
Le calcul suppose **zéro bruit (C = 1)** : il ne garde que les pénalités SWAP et profondeur.

In [ ]:
def build_ghz_linear(n):
    qc = QuantumCircuit(n, n); qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure(range(n), range(n)); return qc

def build_ghz_tree(n):
    qc = QuantumCircuit(n, n); qc.h(0); step = 1
    while step < n:
        for i in range(0, n - step, 2 * step):
            qc.cx(i, i + step)
        step *= 2
    qc.measure(range(n), range(n)); return qc

N0 = candidates[0]
ours = build_ghz(best[N0]["ent"], best[N0]["layers"])
for name, qc in [("cascade", build_ghz_linear(N0)), ("arbre binaire", build_ghz_tree(N0)), ("NOTRE GHZ", ours)]:
    if name == "NOTRE GHZ":
        t = transpile(qc, coupling_map=backend.coupling_map, initial_layout=best[N0]["ent"],
                      optimization_level=3, seed_transpiler=42)
    else:
        t = transpile(qc, coupling_map=backend.coupling_map, optimization_level=3, seed_transpiler=42)
    sw, d = t.count_ops().get("swap", 0), t.depth() - 1
    print(f"{name:14s} SWAP={sw:3d}  profondeur={d:3d}  score max (si C=1) = {challenge_score(N0, 1.0, d, sw):,}")

## 5. Exécution sur la vraie machine
- Chaque circuit est posé **exactement** sur les qubits choisis (`initial_layout`), donc le transpileur n'ajoute aucun SWAP.
- **Twirling de mesure** : il rend l'erreur de lecture symétrique (0→1 aussi probable que 1→0), ce qui est plus simple à corriger.
- **Découplage dynamique (XY4)** : il protège les qubits qui attendent.
- Pour chaque candidat, on ajoute 2 petits circuits de **calibration de lecture**, tous à 0 et tous à 1, sur les mêmes qubits.

In [ ]:
SHOTS = 4000
pubs, tags = [], []
for n in candidates:
    ent, layers = best[n]["ent"], best[n]["layers"]
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend, initial_layout=ent, seed_transpiler=42)
    cal0 = QuantumCircuit(n, n); cal0.measure(range(n), range(n))
    cal1 = QuantumCircuit(n, n); cal1.x(range(n)); cal1.measure(range(n), range(n))
    for kind, qc in (("ghz", build_ghz(ent, layers)), ("cal0", cal0), ("cal1", cal1)):
        isa = pm.run(qc)
        if kind == "ghz":
            n2q = sum(1 for ins in isa.data if ins.operation.num_qubits == 2)
            assert n2q == n - 1, f"le transpileur a ajoute des portes ({n2q} portes 2q au lieu de {n - 1})"
        pubs.append(isa); tags.append((n, kind))

sampler = SamplerV2(mode=backend)
sampler.options.default_shots = SHOTS
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_measure = True
job = sampler.run(pubs)
print("Job envoyé :", job.job_id(), "- en attente de la machine...")
res = job.result()
counts = {tag: r.data.c.get_counts() for tag, r in zip(tags, res)}

## 6. Corrélation brute et corrigée
**Correction de lecture :** on mesure, qubit par qubit, la probabilité de mal lire un 0 (circuit `cal0`) ou un 1 (circuit `cal1`).
La probabilité que les N lectures soient justes est le produit de ces probabilités. On divise donc la part observée de `00…0` par ce produit, et pareil pour `11…1`.
C'est le terme principal de la méthode M3 recommandée par le challenge.

In [ ]:
def flip_rates(cnt, n, prepared):
    # proba, pour chaque qubit, de lire l'inverse de ce qui a ete prepare
    tot = sum(cnt.values()); wrong = [0] * n
    for bits, c in cnt.items():
        for i, b in enumerate(reversed(bits)):   # qiskit : bit 0 a droite
            if b != prepared:
                wrong[i] += c
    return [w / tot for w in wrong]

USE_MITIGATION = True
results = {}
for n in candidates:
    g = counts[(n, "ghz")]; tot = sum(g.values())
    c0, c1 = g.get("0" * n, 0) / tot, g.get("1" * n, 0) / tot
    p10, p01 = flip_rates(counts[(n, "cal0")], n, "0"), flip_rates(counts[(n, "cal1")], n, "1")
    keep0, keep1 = math.prod(1 - p for p in p10), math.prod(1 - p for p in p01)
    raw, mit = c0 + c1, min(1.0, c0 / keep0 + c1 / keep1)
    corr = mit if USE_MITIGATION else raw
    d = best[n]["depth"]
    results[n] = dict(raw=raw, mit=mit, corr=corr, score=challenge_score(n, corr, d))
    print(f"N={n:3d}  C brute={raw:.3f}  C corrigée={mit:.3f}  -> score={results[n]['score']:,}")

N_BEST = max(results, key=lambda n: results[n]["score"])
print("\nMeilleur :", N_BEST, "qubits")

## 7. Prévisualisation officielle du score (fonction du notebook d'origine)

In [ ]:
def preview_challenge_score(circuit: QuantumCircuit, correlation: float):
    num_qubits = circuit.num_qubits
    ops = dict(circuit.count_ops())
    measured_ops = ops.get('measure', 0)
    gate_depth = circuit.depth() - (1 if measured_ops > 0 else 0)
    swap_count = ops.get('swap', 0)
    random_threshold = 1.0 / (2.0 ** (num_qubits - 1))
    base_score = 1000.0 + 50.0 * (num_qubits ** 2 - 4)
    if correlation <= random_threshold:
        print("Correlation sous le bruit aleatoire : score = 0"); return 0
    norm_fidelity = (correlation - random_threshold) / (1.0 - random_threshold)
    fidelity_factor = norm_fidelity ** 2
    swap_penalty = 1.0 / (1.0 + 0.25 * swap_count)
    min_theoretical_depth = 1 + math.ceil(math.log2(num_qubits))
    depth_efficiency = min_theoretical_depth / max(gate_depth, min_theoretical_depth)
    total_score = round(base_score * fidelity_factor * swap_penalty * depth_efficiency)
    print(f"N={num_qubits} | base={base_score:,.0f} | C={correlation:.2%} | fidelite={fidelity_factor:.3f} | "
          f"SWAP={swap_count} | profondeur={gate_depth} (min {min_theoretical_depth}, eff {depth_efficiency:.3f})")
    print(f"SCORE PROJETE : {total_score:,} points")
    return total_score

submission_circuit = build_ghz(best[N_BEST]["ent"], best[N_BEST]["layers"])
submission_correlation = results[N_BEST]["corr"]
preview_challenge_score(submission_circuit, submission_correlation)
display(submission_circuit.draw("mpl", fold=-1))

## 8. Soumission

In [ ]:
API_KEY = "qff_qmLU_KZQPNGmUVR5PthQlv-fRqH8LBS8"
SUBMISSION_URL = "https://qff-26.kodotiq.com/api/submissions"
CHALLENGE_ID = "challenge_beginner"

payload = {"qasm": qasm2.dumps(submission_circuit), "challengeId": CHALLENGE_ID,
           "obtainedResult": float(submission_correlation)}
print(f"Soumission d'un GHZ a {submission_circuit.num_qubits} qubits (job IBM {job.job_id()})...")
try:
    r = requests.post(SUBMISSION_URL, json=payload, headers={"X-API-Key": API_KEY}, timeout=20)
    if r.ok:
        s = r.json()["submission"]; print(f"[OK] id={s['id']}  score officiel={s['score']:,}")
    else:
        print(f"[X] {r.status_code}:", r.json().get("error", r.text))
except Exception as e:
    print("[!] Erreur reseau :", e)